# ANKIT LLM v0.1 — **one-click** Colab T4 run (autopilot)

Tumhe kuch configure nahi karna. Bas **3 cells** chalao.

1. **Cell 1** — GPU check (T4 hona chahiye: *Runtime → Change runtime type → T4 GPU*)
2. **Cell 2** — Drive mount (ek Google login popup aayega, bas OK dabana hai)
3. **Cell 3** — autopilot. Ye khud dekh leta hai:

   * Drive par checkpoint **nahi** hai → poora pipeline: data build, benchmark, auto config,
     pilot, resume test, phir long training.
   * Drive par checkpoint **hai** → seedha training resume (bench/pilot skip — ~8 min bachta hai).

**Checkpoints har 200 steps par Drive par background thread se copy hote hain** — training
Drive ka wait nahi karti, isliye step count aur Drive kabhi zyada peeche nahi rehte.
Runtime band ho jaye to bas yahi cell dobara chalao: newest checkpoint Drive se wapas
aata hai aur wahin se training chalti hai.

Kuch manually edit nahi karna: batch size, dtype, configs — autopilot khud decide karta hai
(benchmark ke *measured* results se).

In [ ]:
!nvidia-smi
import torch, platform
print("py", platform.python_version(), "| torch", torch.__version__)
assert torch.cuda.is_available(), "GPU nahi mila: Runtime -> Change runtime type -> T4 GPU"
p = torch.cuda.get_device_properties(0); cap = torch.cuda.get_device_capability(0)
print("GPU:", p.name, "| cc", f"{cap[0]}.{cap[1]}", "| VRAM", round(p.total_memory/1e9,1), "GB")
print("bf16 accelerated:", cap[0] >= 8, "(T4 -> False, isliye fp16 + GradScaler use hoga)")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = "/content/drive/MyDrive/ankit-llm"
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE}/reports", exist_ok=True)
print("drive ready:", DRIVE)

In [ ]:
# ⬇️ यही एक cell चलाना है (Run ▶). बाकी सब अपने आप होगा.
import os, subprocess, sys

REPO     = "https://github.com/pushparaj9749/tool.git"
REPO_DIR = "/content/ankit-llm"
DRIVE    = "/content/drive/MyDrive/ankit-llm"

def run(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, **kw)

if not os.path.isdir(REPO_DIR):
    run(["git", "clone", REPO, REPO_DIR], check=True)
os.chdir(REPO_DIR)

# --- wo build chuno jisme teeno fixes hain -----------------------------------
# 1. checkpoint device-fix       -> training/checkpoint.py: _to_cpu_copy
# 2. OOM self-heal / VRAM budget -> scripts/colab_autopilot.py: VRAM_BUDGET_FRACTION
# 3. async Drive mirror          -> training/mirror.py: BackgroundMirror
run(["git", "fetch", "origin"], check=False, capture_output=True)
CANDIDATES = ["main", "arena/9852332a-tool"]
MARKERS = {"training/checkpoint.py": "_to_cpu_copy",
           "scripts/colab_autopilot.py": "VRAM_BUDGET_FRACTION",
           "training/mirror.py": "BackgroundMirror",
           "training/pretrain.py": "restore_from_mirror"}

def build_ok() -> tuple[bool, str]:
    if not os.path.exists("scripts/colab_autopilot.py"):
        return False, "autopilot missing"
    missing = []
    for path, marker in MARKERS.items():
        txt = open(path, encoding="utf-8").read() if os.path.exists(path) else ""
        if marker not in txt:
            missing.append(f"{path}:{marker}")
    return (not missing), ("ok" if not missing else "missing " + ", ".join(missing))

chosen = None
for ref in CANDIDATES:
    r = run(["git", "checkout", "-B", "run", f"origin/{ref}"],
            check=False, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"[skip] origin/{ref}: {(r.stderr or '').strip().splitlines()[:1]}")
        continue
    ok, why = build_ok()
    print(f"[{'use' if ok else 'skip'}] origin/{ref}: {why}")
    if ok:
        chosen = ref
        break
assert chosen, ("koi bhi ref async-mirror fix ke saath nahi mila — GitHub par PR "
                "merge karna padega")
print("code build:", chosen, "|", run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())
# data/ aur checkpoints/ gitignored hain -> checkout inhe chhoota nahi

# Colab ka apna torch mat chhedo — sirf chhoti dependencies (pynvml = GPU utilisation)
run([sys.executable, "-m", "pip", "-q", "install",
     "pyyaml", "tokenizers", "numpy", "pynvml", "psutil", "pytest"], check=True)

# --- pehli baar ya resume? Drive dekh kar khud decide hota hai ----------------
def drive_checkpoint_exists() -> bool:
    d = os.path.join(DRIVE, "checkpoints", "t4-full")
    return os.path.isdir(d) and any(f.endswith(".pt") for f in os.listdir(d))

RESUMING = drive_checkpoint_exists()
MODE = "resume" if RESUMING else "all"
print(f"\n>>> MODE = {MODE}  "
      f"({'Drive par checkpoint mila -> seedha training resume' if RESUMING else 'checkpoint nahi -> poora pipeline (bench+pilot+train)'})")

# Colab limit se kam budget (T4 ke liye 480 min; autopilot khud 8 ghante par cap karta hai)
SESSION_BUDGET_MIN = 480
BUILD = "t4fix"   # build-tag: isse pata chalta hai kaunsa code chal raha hai

# Output LIVE Drive me likha jayega — session crash ho jaye tab bhi log bacha rahega
import time
LOG_DIR = os.path.join(DRIVE, "reports")
os.makedirs(LOG_DIR, exist_ok=True)
LOG = os.path.join(LOG_DIR, f"autopilot_{BUILD}_{time.strftime('%m%d_%H%M')}.log")
cmd = [sys.executable, "-u", "scripts/colab_autopilot.py",
       "--mode", MODE, "--session-budget-min", str(SESSION_BUDGET_MIN)]
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("log ->", LOG, "\n", flush=True)
with open(LOG, "w", encoding="utf-8") as fh:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=env)
    for line in proc.stdout:
        print(line, end="", flush=True)
        fh.write(line); fh.flush()
rc = proc.wait()
print(f"\nreturncode = {rc}  |  log saved: {LOG}")
print("\nDrive par ab tak ka newest checkpoint:")
for f in sorted(os.listdir(os.path.join(DRIVE, "checkpoints", "t4-full"))):
    print("  ", f)

## Result dekho

`reports/autopilot_summary.md` (Drive me bhi copy hota hai) me sab measured numbers honge:
benchmark table, auto-selected config, pilot verification, checkpoints, mirror stats.

**Yahi file ka content mujhe bhej dena** — usme sab kuch hoga.

## Chat try karo (GPU par fast hai)

```python
from inference.conversation import ChatEngine
eng = ChatEngine.from_checkpoint("checkpoints/t4-sft/final.pt", device="cuda")
print(eng.chat_once([{"role":"user","content":"नमस्ते अंकित"}], mode="adaptive"))
```

## Agli session (training continue)

Wahi cell 3 dobara chalao. Cell khud `--mode resume` chunta hai: newest checkpoint
Drive se local disk par restore hota hai, phir `--resume auto` usi LR schedule par aage
badhta hai. Sirf pichhle checkpoint ke baad ke kuch minute jaate hain.